# 02. Structural audit (E1–E3)

Metadata only, no model training.

- E1: device field `lrate` vs. labels
- E2: site/sensor overlap in the official train/validation split
- E3: identity-only predictors (in-sample ceiling, official split)

Alarm/normal target: alarm = out, in, noise, other; normal = normal

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from sklearn.metrics import accuracy_score
X1, X1_max, M1, HZ = io_utils.load_d1()
RESULTS = {}

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. E1: device field `lrate`

In [3]:
tab = M1.groupby("cls")[["lrate", "llevel"]].agg(["min", "max", "mean", "std", "nunique"])
print(tab.round(2).to_string())
print("\nlrate value counts by class:")
print(pd.crosstab(M1.lrate, M1.cls).to_string())

truth = np.where(M1.cls.values == "normal", "normal", "alarm")
best = None
for thr in sorted(M1.lrate.unique()):
    pred = np.where(M1.lrate.values >= thr, "alarm", "normal")
    acc = accuracy_score(truth, pred)
    if best is None or acc > best[1]:
        best = (thr, acc)
thr, acc = best
print(f"\nTarget 'alarm vs normal' = {cfg.D1_TASKS['TA_alarm_vs_normal']}")
print(f"Best rule [lrate >= {thr}] -> alarm : accuracy = {acc:.6f}")
print(pd.crosstab(truth, np.where(M1.lrate.values >= thr, "alarm", "normal"),
                  rownames=["truth"], colnames=["rule"]).to_string())

# lrate for other tasks
y2 = np.where(np.isin(M1.cls, ["out", "in"]), "leak", "nonleak")
alarm_only = M1.cls.isin(["out", "in", "noise", "other"]).to_numpy()
within_sensor = M1.groupby("sid").lrate.nunique()
normal_sensor = M1.groupby("sid").cls.first() == "normal"
e1 = dict(
    alarm_target=cfg.D1_TASKS["TA_alarm_vs_normal"],
    rule_threshold=float(thr), rule_accuracy_alarm_vs_normal=float(acc),
    lrate_values_normal=sorted(M1.loc[M1.cls == "normal", "lrate"].unique().tolist()),
    lrate_values_alarm=sorted(M1.loc[M1.cls != "normal", "lrate"].unique().tolist()),
    lrate_constant_within_alarm_classes=bool(M1.loc[alarm_only, "lrate"].nunique() == 1),
    leak_vs_nonleak_rule_accuracy=float(accuracy_score(
        y2, np.where(M1.lrate.values >= thr, "leak", "nonleak"))),
    note_leak_vs_nonleak="lrate = 90 for out/in/noise/other; separates normal rows only",
    sensors_with_varying_lrate=int((within_sensor > 1).sum()),
    of_which_normal_sensors=int(((within_sensor > 1) & normal_sensor).sum()),
    provenance=cfg.D1_PROVIDER_DOC["lrate"],
)
for k, v in e1.items():
    print(f"  {k:38s} {v}")
RESULTS["E1"] = e1
la.save_json(e1, "E1_target_leakage.json")
rp.record("D1.lrate_rule_acc_alarm_vs_normal", acc, "E1_target_leakage.json", "R2-4")

       lrate                          llevel                              
         min max  mean    std nunique    min   max    mean     std nunique
cls                                                                       
in        90  90  90.0   0.00       1    114  1653  504.62  229.50    1037
noise     90  90  90.0   0.00       1    112  1274  586.30  264.93    1051
normal    30  70  45.3  14.11       5    125  1113  276.00  154.43     820
other     90  90  90.0   0.00       1    131  1212  520.87  221.16     932
out       90  90  90.0   0.00       1    134  1254  526.66  224.04    1008

lrate value counts by class:
cls       in  noise  normal  other    out
lrate                                    
30         0      0    7229      0      0
40         0      0    5034      0      0
50         0      0    3807      0      0
60         0      0    3109      0      0
70         0      0    2987      0      0
90     14932   5658       0   7897  19731

Target 'alarm vs normal' = alarm 

## 2. E1b: provider documentation vs. data

Provider state grouping: abnormal = out, in / normal = normal, noise, other (= leak vs. non-leak task)

In [11]:
doc = cfg.D1_PROVIDER_DOC
flagged = M1.cls.isin(["out", "in", "noise", "other"]).to_numpy()
abnormal_state = M1.cls.isin(doc["provider_state_grouping"]["abnormal (이상 상태)"]).to_numpy()
checks = {
    "lrate within documented range 0-90": bool(M1.lrate.between(0, 90).all()),
    "llevel within documented range 0-5000": bool(M1.llevel.between(0, 5000).all()),
    "every flagged record (out/in/noise/other) has lrate = 90": bool((M1.lrate[flagged] == 90).all()),
    "every normal (unflagged) record has lrate < 90": bool((M1.lrate[~flagged] < 90).all()),
    "lrate values observed for normal": sorted(M1.lrate[~flagged].unique().tolist()),
    "lrate values observed for flagged": sorted(M1.lrate[flagged].unique().tolist()),
    "rule lrate>=90 reproduces flagged/unflagged (accuracy)":
        float(np.mean((M1.lrate.values >= 90) == flagged)),
    "rule lrate>=90 vs provider abnormal/normal state (accuracy)":
        float(np.mean((M1.lrate.values >= 90) == abnormal_state)),
    "abnormal-state share (= leak vs non-leak majority)": float(1 - abnormal_state.mean()),
    "lrate separates out vs in": bool(M1.lrate[M1.cls == "out"].nunique() > 1 or
                                      M1.lrate[M1.cls == "in"].nunique() > 1 or
                                      set(M1.lrate[M1.cls == "out"]) != set(M1.lrate[M1.cls == "in"])),
}
reading = (
    "consistent with provider documentation: lrate = 90 for flagged records, < 90 for normal; "
    "no separation of leak vs noise or out vs in"
    if checks["every flagged record (out/in/noise/other) has lrate = 90"]
    and checks["every normal (unflagged) record has lrate < 90"]
    else "NOT consistent with provider documentation - check the table above")
for k, v in checks.items():
    print(f"  {k:62s} {v}")
print("\n" + reading)
la.save_json(dict(provider_documentation=doc, checks=checks, reading=reading),
             "E1_provider_documentation_check.json")
rp.record("D1.lrate_rule_vs_sensor_flag", checks["rule lrate>=90 reproduces flagged/unflagged (accuracy)"],
          "E1_provider_documentation_check.json", "R2-4", note="provider: lrate generated by the sensor")
rp.record("D1.lrate_rule_vs_provider_state", checks["rule lrate>=90 vs provider abnormal/normal state (accuracy)"],
          "E1_provider_documentation_check.json", "R2-4", note="abnormal state = out, in")

  lrate within documented range 0-90                             True
  llevel within documented range 0-5000                          True
  every flagged record (out/in/noise/other) has lrate = 90       True
  every normal (unflagged) record has lrate < 90                 True
  lrate values observed for normal                               [30, 40, 50, 60, 70]
  lrate values observed for flagged                              [90]
  rule lrate>=90 reproduces flagged/unflagged (accuracy)         1.0
  rule lrate>=90 vs provider abnormal/normal state (accuracy)    0.8074136167310753
  abnormal-state share (= leak vs non-leak majority)             0.5075159127074336
  lrate separates out vs in                                      False

consistent with provider documentation: lrate = 90 for flagged records, < 90 for normal; no separation of leak vs noise or out vs in


## 3. E2: overlap in the official train/validation split

Ref.: Kapoor & Narayanan (2023), leakage type L3.2

In [5]:
tr, va = M1[M1.split == "train"], M1[M1.split == "val"]
site_rate = float(va.site.isin(set(tr.site)).mean())
sid_rate  = float(va.sid.isin(set(tr.sid)).mean())
print(f"validation rows whose SITE   appears in training : {site_rate*100:6.2f}%")
print(f"validation rows whose SENSOR appears in training : {sid_rate*100:6.2f}%")
RESULTS["E2"] = dict(site_overlap_rate=site_rate, sid_overlap_rate=sid_rate,
                     n_train=int(len(tr)), n_val=int(len(va)))
la.save_json(RESULTS["E2"], "E2_split_contamination.json")
rp.record("D1.official_val_sensor_overlap", sid_rate, "E2_split_contamination.json", "R1-minor")

validation rows whose SITE   appears in training : 100.00%
validation rows whose SENSOR appears in training :  99.50%


## 4. E3: identity-only predictors

- In-sample lookup = memorisation ceiling (not a signal-free baseline)
- Official provider split: fit on training, score on validation (Table 3)

In [6]:
maj_sid  = M1.groupby("sid").cls.agg(lambda s: s.value_counts().idxmax())
acc_sid  = accuracy_score(M1.cls, M1.sid.map(maj_sid))
maj_site = M1.groupby("site").cls.agg(lambda s: s.value_counts().idxmax())
acc_site = accuracy_score(M1.cls, M1.site.map(maj_site))
base = M1.cls.value_counts(normalize=True).max()

ct = pd.crosstab(M1.sid, M1.cls); purity = (ct.max(axis=1) / ct.sum(axis=1))
print(f"global majority baseline           : {base:.4f}")
print(f"site-identity accuracy (in-sample) : {acc_site:.4f}   <- ceiling")
print(f"sensor-identity accuracy (in-sample): {acc_sid:.4f}   <- ceiling")
print(f"single-class sensors               : {(purity > 0.99).sum()} / {len(purity)}")
RESULTS["E3"] = dict(majority_baseline=float(base),
                     acc_site_identity_insample=float(acc_site),
                     acc_sensor_identity_insample=float(acc_sid),
                     n_pure_sensors=int((purity > 0.99).sum()),
                     n_sensors=int(len(purity)))
la.save_json(RESULTS["E3"], "E3_identity_predictor.json")

trmask = (M1.split == "train").to_numpy()
y5 = M1.cls.to_numpy()
y2 = np.where(np.isin(y5, ["out", "in"]), "leak", "nonleak")
off = []
for task, yy in [("5-class", y5), ("leak/non-leak", y2)]:
    for name, kw in [("sensor id", dict(categorical_cols=["sid"])),
                     ("site id", dict(categorical_cols=["site"])),
                     ("lrate (device)", dict(numeric_cols=["lrate"])),
                     ("all metadata", dict(numeric_cols=["lrate", "llevel"],
                                           categorical_cols=["sid", "site"]))]:
        acc_, base_, n_ = sfb.sfb_official_split(M1, yy, trmask, max_depth=3 if name == "all metadata" else 1,
                                                 **kw)
        off.append(dict(dataset="D1", task=task, predictor=name,
                        protocol="official provider split (train -> val)",
                        sfb=acc_, baseline=base_, n=n_, above_baseline=acc_ - base_))
off = pd.DataFrame(off)
off.to_csv(cfg.RESULTS / "E3_official_split.csv", index=False)
print("\nOfficial provider split (fitted on 'training', scored on 'validation'):")
print(off.round(4).to_string(index=False))
r = off[(off.task == "5-class") & (off.predictor == "sensor id")].iloc[0]
rp.record("D1.official_split.sensor_id_5class", r.sfb, "E3_official_split.csv", "R1-minor(Table3)")

global majority baseline           : 0.3149
site-identity accuracy (in-sample) : 0.6550   <- ceiling
sensor-identity accuracy (in-sample): 1.0000   <- ceiling
single-class sensors               : 2710 / 2710

Official provider split (fitted on 'training', scored on 'validation'):
dataset          task      predictor                               protocol    sfb  baseline    n  above_baseline
     D1       5-class      sensor id official provider split (train -> val) 0.9977    0.3148 7820          0.6829
     D1       5-class        site id official provider split (train -> val) 0.6592    0.3148 7820          0.3444
     D1       5-class lrate (device) official provider split (train -> val) 0.5951    0.3148 7820          0.2803
     D1       5-class   all metadata official provider split (train -> val) 0.6160    0.3148 7820          0.3012
     D1 leak/non-leak      sensor id official provider split (train -> val) 0.9987    0.5075 7820          0.4912
     D1 leak/non-leak        site i